# S6E10 — 08 Richer features for the neural network

**Goal:** extend `X_final` (45 columns) with the extra feature families of the public "Flight Log V3" notebook, to feed a RealMLP (notebook 09). Nothing here uses the competition labels (so no fold logic is needed); the target-encoded keys are built per fold in notebook 09.

**New column families (all computed from the raw columns / the original survey):**
1. **Digits** of the distance and age: `fd_mod10`, `fd_mod100`, `fd_mod1000`, `fd_div10`, `fd_div100`, `fd_length`, `age_mod10` — the generator leaves traces in the last digits.
2. **GPT-2 token keys** of distance, age and the two delays: first token, last token and token count of the number written as text (needs `tiktoken`; the first run downloads the vocabulary — if there is no internet the token block is skipped with a warning).
3. **Delays:** `total_delay`, `delay_recovered`.
4. **Original-survey rates:** for every one of the 21 raw columns, the share of satisfied passengers in the *original* survey among passengers with the same value (`og_rate_*`), and the same for the (travel type, class, customer type) trio. Values absent from the original get the overall original rate.
5. **Second captain:** a CatBoost model trained only on the original survey, its logit for every competition row (`og_cat_logit`). We already have a LightGBM captain (`orig_logit`).
6. **Keys for fold-wise target encoding** (encoded inside each fold in notebook 09): `key_age_class`, `key_fd_class`, `key_age_travel`.

**Output:** `data/processed/X_rich_train.parquet`, `X_rich_test.parquet` (X_final + the new columns).

In [1]:
import re, time, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')
SMOKE = False      # True: CatBoost captain with 100 trees (a few seconds)

RAW = next(p for p in [Path('data/raw'), Path('../data/raw'), Path('/kaggle/input/playground-series-s6e10')] if (p / 'train.csv').exists())
PROC = next(p for p in [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')] if (p / 'X_final_train.parquet').exists())
ORIG = next(p for p in [RAW / 'orig_data.csv', Path('/kaggle/input/s6e10-orig-data/orig_data.csv')] if p.exists())

train = pd.read_csv(RAW / 'train.csv'); test = pd.read_csv(RAW / 'test.csv'); orig = pd.read_csv(ORIG)
Xf_tr = pd.read_parquet(PROC / 'X_final_train.parquet'); Xf_te = pd.read_parquet(PROC / 'X_final_test.parquet')
assert len(train) == len(Xf_tr) == 699635 and len(test) == len(Xf_te) == 299844
assert (train['Age'].values == Xf_tr['age'].values).all() and (train['Flight Distance'].values == Xf_tr['flight_distance'].values).all()
assert (test['Age'].values == Xf_te['age'].values).all() and (test['Flight Distance'].values == Xf_te['flight_distance'].values).all()
print('rows aligned with X_final | original survey rows:', len(orig))

def tobin(s):
    return s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'satisfied']).astype(int)
y_orig = tobin(orig['satisfaction'])
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [c for c in test.columns if c not in CATS + NUMS + ['id']]
BASE = NUMS + RATINGS + CATS
assert len(RATINGS) == 13
clean = lambda s: re.sub(r'[^0-9a-zA-Z]+', '_', s).strip('_').lower()
full = pd.concat([train[BASE], test[BASE]], ignore_index=True)
n_train = len(train)
E = pd.DataFrame(index=full.index)          # the new columns

rows aligned with X_final | original survey rows: 129880


## 1. Digits and delays

In [2]:
fd, age = full['Flight Distance'], full['Age']
E['fd_mod10'], E['fd_mod100'], E['fd_mod1000'] = fd % 10, fd % 100, fd % 1000
E['fd_div10'], E['fd_div100'], E['age_mod10'] = fd // 10, fd // 100, age % 10
E['fd_length'] = fd.astype(int).astype(str).str.len()
dep, arr = full['Departure Delay in Minutes'], full['Arrival Delay in Minutes']
arr_f = arr.fillna(dep)
E['total_delay'] = dep + arr_f
E['delay_recovered'] = dep - arr_f
print('digits + delays:', E.shape[1], 'columns')

digits + delays: 9 columns


## 2. GPT-2 token keys

In [3]:
try:
    import tiktoken
    GPT2 = tiktoken.get_encoding('gpt2')
    for col, nm in [('Flight Distance', 'fd'), ('Age', 'age'), ('Departure Delay in Minutes', 'dep'), ('Arrival Delay in Minutes', 'arr')]:
        toks = {v: GPT2.encode(' ' + str(int(v))) for v in full[col].dropna().unique()}
        E[f'{nm}_tok_first'] = full[col].map({v: t[0] for v, t in toks.items()})
        E[f'{nm}_tok_last'] = full[col].map({v: t[-1] + 100_000 * len(t) for v, t in toks.items()})
        E[f'{nm}_tok_count'] = full[col].map({v: len(t) for v, t in toks.items()})
    print('token keys added; example distance 3983 ->', GPT2.encode(' 3983'))
except Exception as err:
    print('WARNING: GPT-2 token keys skipped:', str(err)[:150])

token keys added; example distance 3983 -> [39260, 18]


## 3. Original-survey rates (no competition labels)

In [4]:
glob_rate = y_orig.mean()
for col in BASE:
    rate = y_orig.groupby(orig[col].values).mean()
    E[f'og_rate_{clean(col)}'] = full[col].map(rate).fillna(glob_rate)
trio = ['Type of Travel', 'Class', 'Customer Type']
trio_rate = orig[trio].assign(y=y_orig.values).groupby(trio)['y'].mean().rename('trio_og_rate')
E['og_rate_trio'] = full[trio].merge(trio_rate, left_on=trio, right_index=True, how='left')['trio_og_rate'].fillna(glob_rate).values
print('original-survey rates:', sum(c.startswith('og_rate') for c in E.columns), 'columns | share of distances found in the original:',
      round(full['Flight Distance'].isin(set(orig['Flight Distance'])).mean(), 4))

original-survey rates: 22 columns | share of distances found in the original: 1.0


## 4. Second captain: CatBoost trained only on the original survey

In [5]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
CAT_MAPS = {c: {v: i for i, v in enumerate(sorted(pd.concat([train[c], test[c]]).dropna().unique()))} for c in CATS}
def enc(df):
    X = df[BASE].copy()
    for c in CATS: X[c] = X[c].map(CAT_MAPS[c]).fillna(-1).astype('int16')
    return X
og = orig.dropna(subset=[c for c in BASE if c != 'Arrival Delay in Minutes'])[BASE + ['satisfaction']].drop_duplicates()
t0 = time.time()
cb = CatBoostClassifier(iterations=100 if SMOKE else 1500, learning_rate=0.05, depth=8, cat_features=CATS, random_seed=42, verbose=0,
                        allow_writing_files=False, thread_count=-1)
cb.fit(enc(og), tobin(og['satisfaction']).values)
p = cb.predict_proba(enc(full))[:, 1].clip(1e-5, 1 - 1e-5)
E['og_cat_logit'] = np.log(p / (1 - p))
y_tr = tobin(train['satisfaction']).values
print(f'captain trained on {len(og):,} original rows in {time.time()-t0:.0f}s | AUC on competition train (no labels used): {roc_auc_score(y_tr, p[:n_train]):.4f}  (the LightGBM captain gives 0.95472)')

captain trained on 129,880 original rows in 62s | AUC on competition train (no labels used): 0.9544  (the LightGBM captain gives 0.95472)


## 5. Keys for fold-wise target encoding, assemble and save

In [6]:
cls = full['Class'].map(CAT_MAPS['Class']); trv = full['Type of Travel'].map(CAT_MAPS['Type of Travel'])
E['key_age_class'] = age * 10 + cls
E['key_fd_class'] = fd * 10 + cls
E['key_age_travel'] = age * 10 + trv

E = E.fillna(-1).astype('float32')     # NaN only where a delay is missing (token columns); -1 = missing, the flag arrival_delay_missing already exists
Etr, Ete = E.iloc[:n_train].reset_index(drop=True), E.iloc[n_train:].reset_index(drop=True)
Xr_tr = pd.concat([Xf_tr.reset_index(drop=True), Etr], axis=1); Xr_te = pd.concat([Xf_te.reset_index(drop=True), Ete], axis=1)
assert not Xr_tr.isna().any().any() or print('NaN columns:', Xr_tr.columns[Xr_tr.isna().any()].tolist())
Xr_tr.to_parquet(PROC / 'X_rich_train.parquet'); Xr_te.to_parquet(PROC / 'X_rich_test.parquet')
print('saved X_rich:', Xr_tr.shape, Xr_te.shape, '| new columns:', E.shape[1]); print(list(E.columns))

saved X_rich: (699635, 92) (299844, 92) | new columns: 47
['fd_mod10', 'fd_mod100', 'fd_mod1000', 'fd_div10', 'fd_div100', 'age_mod10', 'fd_length', 'total_delay', 'delay_recovered', 'fd_tok_first', 'fd_tok_last', 'fd_tok_count', 'age_tok_first', 'age_tok_last', 'age_tok_count', 'dep_tok_first', 'dep_tok_last', 'dep_tok_count', 'arr_tok_first', 'arr_tok_last', 'arr_tok_count', 'og_rate_age', 'og_rate_flight_distance', 'og_rate_departure_delay_in_minutes', 'og_rate_arrival_delay_in_minutes', 'og_rate_inflight_wifi_service', 'og_rate_departure_arrival_time_convenient', 'og_rate_ease_of_online_booking', 'og_rate_gate_location', 'og_rate_food_and_drink', 'og_rate_online_boarding', 'og_rate_seat_comfort', 'og_rate_inflight_entertainment', 'og_rate_on_board_service', 'og_rate_leg_room_service', 'og_rate_baggage_handling', 'og_rate_checkin_service', 'og_rate_cleanliness', 'og_rate_gender', 'og_rate_customer_type', 'og_rate_type_of_travel', 'og_rate_class', 'og_rate_trio', 'og_cat_logit', 'key

Next: `09_realmlp_rich.ipynb`.